In [ ]:
import pandas as pd
from chembl_webresource_client.new_client import new_client

groups = {
    "Group 1 - 5HT2A selective": [
        "risperidone",
        "ketanserin",
        "clozapine",
        "volinanserin",
        "eplivanserin",
        "altanserin"
],

    "Group 1 - D2 selective": [
        "raclopride",
        "sulpiride",
        "amisulpride",
        "L-741626",
        "eticlopride",
        "nemonapride",
        "remoxipride"
],

    "Group 2 - low activity controls": [
        "atropine",
        "naloxone",
        "caffeine",
        "scopolamine",
        "nicotine",
        "acetazolamide",
        "furosemide"
    ],

    "Group 3 - non-selective / mixed": [
        "spiperone",
        "pimozide",
        "thioridazine",
        "haloperidol",
        "chlorpromazine",
        "olanzapine"
    ]
}

#grupa 1: Ligandy wykazujące preferencyjne wiązanie z receptorem 5-HT2A
#grupa 1: Ligandy wykazujące preferencyjne wiązanie z receptorem dopaminowym D2
#grupa 2: Związki o odmiennych głównych celach biologicznych, wykorzystane jako grupa kontrolna
#grupa 3: Ligandy o szerokim profilu receptorowym, oddziałujące na wiele receptorów jednocześnie:

targets = {
    "D2": "CHEMBL217",
    "5HT2A": "CHEMBL224"
}

molecule = new_client.molecule
activity = new_client.activity

def find_molecule(compound_name):
    """Find first ChEMBL molecule matching compound name."""
    results = molecule.search(compound_name)

    if len(results) == 0:
        return None

    mol = results[0]

    return {
        "query_name": compound_name,
        "chembl_id": mol.get("molecule_chembl_id"),
        "pref_name": mol.get("pref_name"),
        "canonical_smiles": (
            mol.get("molecule_structures", {}) or {}
        ).get("canonical_smiles")
    }


def get_ki_values(molecule_chembl_id, target_chembl_id):
    """Get Ki values in nM for one molecule-target pair."""
    acts = activity.filter(
        molecule_chembl_id=molecule_chembl_id,
        target_chembl_id=target_chembl_id,
        standard_type="Ki"
    )

    rows = []

    for a in acts:
        value = a.get("standard_value")
        units = a.get("standard_units")
        relation = a.get("standard_relation")

        if value is not None:
            rows.append({
                "standard_value": float(value),
                "standard_units": units,
                "standard_relation": relation,
                "assay_description": a.get("assay_description"),
                "document_chembl_id": a.get("document_chembl_id")
            })

    return rows


def summarize_ki(ki_rows):
    """Return median Ki and number of measurements."""
    if len(ki_rows) == 0:
        return None, 0

    values = [r["standard_value"] for r in ki_rows]

    return pd.Series(values).median(), len(values)

final_rows = []

for group_name, compound_names in groups.items():
    for compound_name in compound_names:

        mol_info = find_molecule(compound_name)

        if mol_info is None:
            final_rows.append({
                "group": group_name,
                "query_name": compound_name,
                "chembl_id": None,
                "pref_name": None,
                "canonical_smiles": None,
                "Ki_D2_nM_median": None,
                "Ki_D2_n": 0,
                "Ki_5HT2A_nM_median": None,
                "Ki_5HT2A_n": 0,
                "selectivity_D2_over_5HT2A": None,
                "selectivity_5HT2A_over_D2": None
            })
            continue

        d2_ki_rows = get_ki_values(mol_info["chembl_id"], targets["D2"])
        htr2a_ki_rows = get_ki_values(mol_info["chembl_id"], targets["5HT2A"])

        d2_median, d2_n = summarize_ki(d2_ki_rows)
        htr2a_median, htr2a_n = summarize_ki(htr2a_ki_rows)

        if d2_median is not None and htr2a_median is not None:
            selectivity_d2 = htr2a_median / d2_median
            selectivity_5ht2a = d2_median / htr2a_median
        else:
            selectivity_d2 = None
            selectivity_5ht2a = None

        final_rows.append({
            "group": group_name,
            "query_name": compound_name,
            "chembl_id": mol_info["chembl_id"],
            "pref_name": mol_info["pref_name"],
            "canonical_smiles": mol_info["canonical_smiles"],
            "Ki_D2_nM_median": d2_median,
            "Ki_D2_n": d2_n,
            "Ki_5HT2A_nM_median": htr2a_median,
            "Ki_5HT2A_n": htr2a_n,
            "selectivity_D2_over_5HT2A": selectivity_d2,
            "selectivity_5HT2A_over_D2": selectivity_5ht2a
        })

df = pd.DataFrame(final_rows)

df

,group,query_name,chembl_id,pref_name,canonical_smiles,Ki_D2_nM_median,Ki_D2_n,Ki_5HT2A_nM_median,Ki_5HT2A_n,selectivity_D2_over_5HT2A,selectivity_5HT2A_over_D2
0,Group 1 - 5HT2A selective,risperidone,CHEMBL85,RISPERIDONE,Cc1nc2n(c(=O)c1CCN1CCC(c3noc4cc(F)ccc34)CC1)CCCC2,3.850,20,0.50,22,0.129870,7.700000
1,Group 1 - 5HT2A selective,ketanserin,CHEMBL1256709,KETANSERIN TARTRATE,O=C(O)C(O)C(O)C(=O)O.O=C(c1ccc(F)cc1)C1CCN(CCn...,NaN,0,NaN,0,NaN,NaN
2,Group 1 - 5HT2A selective,clozapine,CHEMBL1688,NaN,C[N+]1([O-])CCN(C2=Nc3cc(Cl)ccc3Nc3ccccc32)CC1,NaN,0,NaN,0,NaN,NaN
3,Group 1 - 5HT2A selective,volinanserin,CHEMBL74355,VOLINANSERIN,COc1cccc([C@H](O)C2CCN(CCc3ccc(F)cc3)CC2)c1OC,1300.000,4,0.31,9,0.000238,4193.548387
4,Group 1 - 5HT2A selective,eplivanserin,CHEMBL2103845,EPLIVANSERIN FUMARATE,CN(C)CCO/N=C(/C=C/c1ccc(O)cc1)c1ccccc1F.CN(C)C...,NaN,0,NaN,0,NaN,NaN
5,Group 1 - 5HT2A selective,altanserin,CHEMBL1371857,NaN,Cl.O.O=C(c1ccc(F)cc1)C1CCN(CCn2c(=S)[nH]c3cccc...,NaN,0,NaN,0,NaN,NaN
6,Group 1 - D2 selective,raclopride,CHEMBL2104769,RACLOPRIDE C 11,CCN1CCC[C@H]1CNC(=O)c1c(O)c(Cl)cc(Cl)c1O[11CH3],NaN,0,NaN,0,NaN,NaN
7,Group 1 - D2 selective,sulpiride,CHEMBL26,SULPIRIDE,CCN1CCCC1CNC(=O)c1cc(S(N)(=O)=O)ccc1OC,38.900,4,NaN,0,NaN,NaN
8,Group 1 - D2 selective,amisulpride,CHEMBL243712,AMISULPRIDE,CCN1CCCC1CNC(=O)c1cc(S(=O)(=O)CC)c(N)cc1OC,7.795,2,630.96,1,80.944195,0.012354
9,Group 1 - D2 selective,L-741626,CHEMBL58832,ASPARAGINE,NC(=O)C[C@H](N)C(=O)O,NaN,0,NaN,0,NaN,NaN
